In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely import wkt
import plotly.express as px

In [30]:
data = gpd.read_file("Etablissements_santé.geojson")
data.head()

,type,name,operator,emergency,wheelchair,opening_hours,capacity,for_public,phone,ref_finess,...,meta_code_dep,meta_name_reg,meta_code_reg,meta_osm_id,meta_osm_url,meta_first_update,meta_last_update,meta_versions_number,meta_users_number,geometry
0,nursing_home,Ehpad du CH,None,None,None,None,None,senior,+33385884493,710013509,...,71,Bourgogne-Franche-Comté,27,7696263271,https://www.openstreetmap.org/node/7696263271,2020-07-08 21:24:22+00:00,2022-08-18,2.0,2.0,POINT (4.1241 46.45343)
1,nursing_home,Ehpad du CH,None,None,None,None,None,senior,+33474065250,690787510,...,69,Auvergne-Rhône-Alpes,84,7701544666,https://www.openstreetmap.org/node/7701544666,2020-07-10 07:47:21+00:00,2022-08-18,2.0,2.0,POINT (4.74738 46.11017)
2,nursing_home,Ehpad du CH,None,None,None,None,None,senior,+33385796900,710972977,...,71,Bourgogne-Franche-Comté,27,7701680916,https://www.openstreetmap.org/node/7701680916,2020-07-10 08:43:03+00:00,2022-08-18,2.0,2.0,POINT (4.13717 46.6923)
3,dentist,Dr Loudmilla Samya Oukaci,None,None,None,None,None,None,None,None,...,57,Grand Est,44,7722363124,https://www.openstreetmap.org/node/7722363124,2020-07-16 23:15:36+00:00,2023-08-28,2.0,1.0,POINT (6.17879 49.09646)
4,nursing_home,EHPAD Les Orchidées,SAS les Orchidées,None,None,None,27,senior,+33 4 93 40 54 64,060799020,...,06,Provence-Alpes-Côte d'Azur,93,7723453687,https://www.openstreetmap.org/node/7723453687,2020-07-17 07:46:50+00:00,2022-08-18,2.0,2.0,POINT (6.92928 43.65897)


In [31]:
codes_grenoble = [
    "38057","38059","38068","38071","38111","38126","38150","38151","38158",
    "38169","38170","38179","38185","38187","38188","38200","38229","38235",
    "38252","38258","38271","38277","38279","38281","38309","38317","38325",
    "38328","38364","38382","38388","38421","38423","38436","38445","38471",
    "38472","38474","38478","38485","38486","38516","38524","38528","38529",
    "38533","38540","38545","38562"
]
codes_rennes = [
    "35001","35022","35024","35032","35039","35047","35051","35055","35058",
    "35059","35065","35066","35076","35079","35080","35081","35088","35120",
    "35131","35139","35144","35180","35189","35196","35204","35206","35208",
    "35210","35216","35238","35240","35245","35250","35266","35285","35288",
    "35315","35334","35351","35352","35353","35355","35363"
]
codes_rouen = [
    "76005", "76018", "76056", "76066", "76084", "76095", "76108", "76111", "76115",
    "76157", "76165", "76178", "76200", "76212", "76222", "76231", "76237", "76263",
    "76284", "76286", "76309", "76319", "76352", "76354", "76362", "76377", "76378",
    "76119", "76382", "76466", "76322", "76366", "76429", "76431", "76541", "76673",
    "76052", "76402", "76410", "76451", "76448", "76453", "76474", "76484", "76486",
    "76497", "76517", "76524", "76544", "76540", "76550", "76558", "76560", "76561",
    "76575", "76591", "76598", "76611", "76617", "76631", "76634", "76636", "76640",
    "76582", "76681", "76682", "76705", "76717", "76752", "76753", "76754"
]
codes_saint_etienne = [
    "42001", "42005", "42031", "42044", "42051", "42053", "42058", "42083", "42085",
    "42094", "42095", "42098", "42101", "42103", "42110", "42115", "42097", "42100",
    "42106", "42183", "42306", "42307", "42311", "42322", "42037", "42125", "42133",
    "42168", "42186", "42188", "42191", "42211", "42207", "42208", "42218", "42222",
    "42223", "42234", "42243", "42242", "42259", "42262", "42266", "42270", "42271",
    "42275", "42281", "42210", "42302", "42309", "42317", "42323", "42330"
]
codes_montpellier = [
    "34022", "34027", "34057", "34058", "34077", "34087", "34088", "34095", "34116",
    "34120", "34123", "34129", "34134", "34145", "34164", "34169", "34172", "34179", 
    "34198", "34202", "34217", "34236", "34244", "34249", "34276", "34259", "34270",
    "34295", "34307", "34327", "34337"
]
data_filtre = data[data["meta_code_com"].isin(codes_grenoble + codes_rennes + codes_rouen + codes_saint_etienne + codes_montpellier)]

In [32]:
colonnes_a_supprimer = [
    'operator', 'emergency', 'wheelchair', 'opening_hours',
    'capacity', 'for_public', 'phone', 'ref_finess', 'type_finess',
    'ref_naf', 'ref_siret', 'meta_osm_id', 'meta_osm_url',
    'meta_first_update', 'meta_last_update',
    'meta_versions_number', 'meta_users_number'
]
data_filtre = data_filtre.drop(columns=colonnes_a_supprimer)

In [33]:
data_filtre = data_filtre.rename(columns={
    "type": "type_etablissement",
    "name": "nom_etablissement",
    "meta_name_com": "commune",
    "meta_code_com": "code_commune",
    "meta_name_dep": "departement",
    "meta_code_dep": "code_departement",
    "meta_name_reg": "region",
    "meta_code_reg": "code_region",
})
conditions = [
    data_filtre["code_commune"].isin(codes_grenoble),
    data_filtre["code_commune"].isin(codes_rennes),
    data_filtre["code_commune"].isin(codes_rouen),
    data_filtre["code_commune"].isin(codes_saint_etienne),
    data_filtre["code_commune"].isin(codes_montpellier)
]

choix = ["Grenoble", "Rennes", "Rouen", "Saint-Étienne", "Montpellier"]

data_filtre["Métropole"] = np.select(conditions, choix, default="Autre")
data_filtre.head()

,type_etablissement,nom_etablissement,commune,code_commune,departement,code_departement,region,code_region,geometry,Métropole
47,dentist,Olivier Quéré,Rennes,35238,Ille-et-Vilaine,35,Bretagne,53,POINT (-1.6634 48.09456),Rennes
65,hospital,Hôpital de jour Addictologie,Saint-Martin-d'Hères,38421,Isère,38,Auvergne-Rhône-Alpes,84,POINT (5.75351 45.185),Grenoble
86,pharmacy,Pharmacie Jaouen,Vezin-le-Coquet,35353,Ille-et-Vilaine,35,Bretagne,53,POINT (-1.75734 48.11907),Rennes
88,doctors,Kiné-Santé Pédicure Podologue,Saint-Chamond,42207,Loire,42,Auvergne-Rhône-Alpes,84,POINT (4.50816 45.47309),Saint-Étienne
99,pharmacy,Pharmacie Grangeon,Échirolles,38151,Isère,38,Auvergne-Rhône-Alpes,84,POINT (5.71883 45.15646),Grenoble


In [34]:
gdf = gpd.GeoDataFrame(data_filtre, geometry="geometry", crs="EPSG:4326")
gdf["lon"] = gdf.geometry.x
gdf["lat"] = gdf.geometry.y
grenoble = gdf[gdf["Métropole"] == "Grenoble"]
fig_map = px.scatter_mapbox(
    grenoble,
    lat="lat",
    lon="lon",
    hover_name="nom_etablissement",
    hover_data={
        "type_etablissement": True,
        "commune": True,
        "nom_etablissement": False
    },
    color="type_etablissement",
    zoom=10,
    title="Établissements de santé - Métropole de Grenoble",
    height=600
)
fig_map.update_layout(
    mapbox_style="carto-positron",  # 👈 fond noir & blanc
    template="plotly_white",
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)
fig_map.show()

/var/folders/_1/1lx811qj7f19ynqlp2p83tcw0000gn/T/ipykernel_6919/341224695.py:5: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_map = px.scatter_mapbox(


In [35]:
data_filtre.to_file("Etablissements_santé_filtre.geojson", driver="GeoJSON")

In [76]:
election_2014 = pd.read_csv("Elections/MN14_Bvot_T1T2.txt", sep=";", encoding="latin1")
election_2014.head()

/var/folders/_1/1lx811qj7f19ynqlp2p83tcw0000gn/T/ipykernel_6919/2048159581.py:1: DtypeWarning:

Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.



,N° tour,Code département,Code commune,Nom de la commune,N° de bureau de vote,Inscrits,Votants,Exprimés,N° de dépôt de la liste,Nom du candidat tête de liste,Prénom du candidat tête de liste,Code nuance de la liste,Nombre de voix
0,1,1,1,L'Abergement-Clémenciat,0001,599,355,341,4,BOUILLOUX,Delphine,NC,324
1,1,1,1,L'Abergement-Clémenciat,0001,599,355,341,16,EVALET-TAPONAT,Line,NC,316
2,1,1,1,L'Abergement-Clémenciat,0001,599,355,341,3,BERAUD,Zélie,NC,315
3,1,1,1,L'Abergement-Clémenciat,0001,599,355,341,14,VACLE,Robert,NC,300
4,1,1,1,L'Abergement-Clémenciat,0001,599,355,341,10,MARGUIN,Jean Paul,NC,276


In [77]:
election_1er_tour_2020 = pd.read_excel("Elections/Election_1er_tour.xlsx")
election_2eme_tour_2020 = pd.read_excel("Elections/Election_2eme_tour.xlsx")
election_1er_tour_2020.head()

,Code du département,Libellé du département,Code de la commune,Libellé de la commune,Code B.Vote,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,...,% Voix/Exp.60,N.Pan..61,Code Nuance.61,Sexe.61,Nom.61,Prénom.61,Liste.61,Voix.61,% Voix/Ins.61,% Voix/Exp.61
0,1,Ain,1,L'Abergement-Clémenciat,1,622,340,54.66,282,45.34,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Ain,2,L'Abergement-de-Varey,1,212,83,39.15,129,60.85,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Ain,4,Ambérieu-en-Bugey,1,1042,686,65.83,356,34.17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Ain,4,Ambérieu-en-Bugey,2,1064,722,67.86,342,32.14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,Ain,4,Ambérieu-en-Bugey,3,1120,696,62.14,424,37.86,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [78]:
communes_grenoble = [
    "Bresson","Brié-et-Angonnes","Champ-sur-Drac","Champagnier","Claix",
    "Corenc","Domène","Echirolles","Eybens","Fontaine","Fontanil-Cornillon",
    "Gières","Grenoble","Herbeys","Jarrie","La Tronche","Le Gua",
    "Le Pont-de-Claix","Le Sappey-en-Chartreuse","Meylan","Miribel-Lanchâtre",
    "Mont-Saint-Martin","Montchaboud","Murianette","Notre-Dame-de-Commiers",
    "Notre-Dame-de-Mésage","Noyarey","Poisat","Proveysieux","Quaix-en-Chartreuse",
    "Saint-Barthélemy-de-Séchilienne","Saint-Egrève","Saint-Georges-de-Commiers",
    "Saint-Martin-d'Hères","Saint-Martin-le-Vinoux","Saint-Paul-de-Varces",
    "Saint-Pierre-de-Mésage","Sarcenas","Sassenage","Séchilienne",
    "Seyssinet-Pariset","Seyssins","Varces-Allières-et-Risset",
    "Vaulnaveys-le-Bas","Vaulnaveys-le-Haut","Venon","Veurey-Voroize",
    "Vif","Vizille"
]

communes_rennes = [
    "Acigné", "Bécherel","Betton","Bourgbarré","Brécé","Bruz","Cesson-Sévigné",
    "Chantepie","Chartres-de-Bretagne","Chavagne","Chevaigné","Cintré","Corps-Nuds","Gévezé",
    "La Chapelle-des-Fougeretz","La Chapelle-Thouarault","L'Hermitage","Le Rheu","Le Verger",
    "Montgermont","Mordelles","Noyal-Châtillon-sur-Seiche","Nouvoitou","Orgères",
    "Pacé","Parthenay-de-Bretagne","Pont-Péan","Rennes","Romillé","Saint-Armel","Saint-Erblon",
    "Saint-Gilles","Saint-Grégoire","Saint-Jacques-de-la-Lande","Saint-Sulpice-la-Forêt",
    "Thorigné-Fouillard","Vern-sur-Seiche","Vezin-le-Coquet", "Clayes", "La Chapelle-Chaussée", 
    "Laillé", "Langan", "Miniac-sous-Bécherel"
]

communes_rouen = [
    "Amfreville-la-Mi-Voie","Anneville-Ambourville","Bardouville","Belbeuf",
    "Berville-sur-Seine","Bihorel","Bois-Guillaume","Bonsecours","Boos",
    "Canteleu","Caudebec-lès-Elbeuf","Cléon","Darnétal","Déville-lès-Rouen",
    "Duclair","Elbeuf","Epinay-sur-Duclair","Fontaine-sous-Préaux",
    "Franqueville-Saint-Pierre","Freneuse","Gouy","Grand-Couronne",
    "Hautot-sur-Seine","Hénouville","Houppeville","Isneauville","Jumièges",
    "La Bouille","La Londe","La Neuville-Chant-d'Oisel","Le Grand-Quevilly",
    "Le Houlme","Le Mesnil-Esnard","Le Mesnil-sous-Jumièges","Le Petit-Quevilly",
    "Le Trait","Les Authieux-sur-le-Port-Saint-Ouen","Malaunay","Maromme",
    "Mont-Saint-Aignan","Montmain","Moulineaux","Notre-Dame-de-Bondeville",
    "Oissel","Orival","Petit-Couronne","Quevillon",
    "Quévreville-la-Poterie","Roncherolles-sur-le-Vivier","Rouen","Sahurs",
    "Saint-Aubin-Celloville","Saint-Aubin-Epinay","Saint-Aubin-lès-Elbeuf",
    "Saint-Etienne-du-Rouvray","Saint-Jacques-sur-Darnétal",
    "Saint-Léger-du-Bourg-Denis","Saint-Martin-de-Boscherville",
    "Saint-Martin-du-Vivier","Saint-Paër","Saint-Pierre-de-Manneville",
    "Saint-Pierre-de-Varengeville","Saint-Pierre-lès-Elbeuf",
    "Sainte-Marguerite-sur-Duclair","Sotteville-lès-Rouen",
    "Sotteville-sous-le-Val","Tourville-la-Rivière","Val-de-la-Haye",
    "Yainville","Ymare","Yville-sur-Seine"
]

communes_saint_etienne = [
    "Aboën","Andrézieux-Bouthéon","Caloire","Cellieu","Chagnon","Chamboeuf",
    "Châteauneuf","Dargoire","Doizieux","Farnay","Firminy","Fontanès",
    "Fraisses","Genilac","L'Etrat","L'Horme","La Fouillouse","La Gimond",
    "La Grand-Croix","La Ricamarie","La Talaudière","La Terrasse-sur-Dorlay",
    "La Tour-en-Jarez","La Valla-en-Gier","Le Chambon-Feugerolles","Lorette",
    "Marcenod","Pavezin","Rive-de-Gier","Roche-la-Molière",
    "Rozier-Côtes-d'Aurec","Saint-Bonnet-les-Oules","Saint-Chamond",
    "Saint-Christo-en-Jarez","Saint-Etienne","Saint-Galmier",
    "Saint-Genest-Lerpt","Saint-Héand","Saint-Jean-Bonnefonds","Saint-Joseph",
    "Saint-Martin-la-Plaine","Saint-Maurice-en-Gourgois","Saint-Nizier-de-Fornas",
    "Saint-Paul-en-Cornillon","Saint-Paul-en-Jarez","Saint-Priest-en-Jarez",
    "Saint-Romain-en-Jarez","Sainte-Croix-en-Jarez","Sorbiers","Tartaras",
    "Unieux","Valfleury","Villars"
]

communes_montpellier = [
    "Baillargues","Beaulieu","Castelnau-le-Lez","Castries","Clapiers",
    "Cournonsec","Cournonterral","Fabrègues","Grabels","Jacou","Juvignac",
    "Lattes","Lavérune","Le Crès","Montaud","Montferrier-sur-Lez",
    "Montpellier","Murviel-lès-Montpellier","Pérols","Pignan","Prades-le-Lez",
    "Restinclières","Saint-Brès","Saint-Drézéry","Saint-Geniès-des-Mourgues",
    "Saint-Georges-d'Orques","Saint-Jean-de-Védas","Saussan","Sussargues",
    "Vendargues","Villeneuve-lès-Maguelone"
]

mapping_metropoles = {
    "Isère": communes_grenoble,
    "Ille-et-Vilaine": communes_rennes,
    "Seine-Maritime": communes_rouen,
    "Loire": communes_saint_etienne,
    "Hérault": communes_montpellier
}

def filtrer_metropoles(df):
    masque = False
    for dept, communes in mapping_metropoles.items():
        condition = (df["Libellé du département"] == dept) & (df["Libellé de la commune"].isin(communes))
        masque = masque | condition
    return df[masque].loc[:, : "% Exp/Vot"]

df_tour1_2020 = filtrer_metropoles(election_1er_tour_2020)
df_tour1_2020["Année"] = 2020
df_tour1_2020["Numéro de tour"] = 1

df_tour2_2020 = filtrer_metropoles(election_2eme_tour_2020)
df_tour2_2020["Année"] = 2020
df_tour2_2020["Numéro de tour"] = 2

election_2020 = pd.concat([df_tour1_2020, df_tour2_2020], ignore_index=True)

comptage_com_2020 = election_2020.groupby("Libellé du département")["Libellé de la commune"].nunique().reset_index()
comptage_com_2020.columns = ["Département", "Nombre de communes uniques"]
print(comptage_com_2020)

mapping_codes_metropoles = {
    38: communes_grenoble,
    35: communes_rennes,
    76: communes_rouen,
    42: communes_saint_etienne,
    34: communes_montpellier
}

def filtrer_codes_metropoles(df):
    masque = False
    for dept, communes in mapping_codes_metropoles.items():
        condition = (df["Code département"] == dept) & (df["Nom de la commune"].isin(communes))
        masque = masque | condition
    return df[masque].loc[:, : "Exprimés"]

election_2014 = filtrer_codes_metropoles(election_2014)

comptage_com_2014 = election_2014.groupby("Code département")["Nom de la commune"].nunique().reset_index()
comptage_com_2014.columns = ["Département", "Nombre de communes uniques"]
print(comptage_com_2014)

       Département  Nombre de communes uniques
0          Hérault                          31
1  Ille-et-Vilaine                          43
2            Isère                          49
3            Loire                          53
4   Seine-Maritime                          71
   Département  Nombre de communes uniques
0           34                          31
1           35                          43
2           38                          49
3           42                          53
4           76                          71


In [80]:
departements = {
    34: "Hérault", 35: "Ille-et-Vilaine",
    38: "Isère", 42: "Loire", 76: "Seine-Maritime"
}

election_2014 = election_2014.rename(columns={
    "Code département": "Code du département",
    "Code commune": "Code de la commune",
    "Nom de la commune": "Libellé de la commune",
    "N° de bureau de vote": "Code B.Vote",
    "N° tour": "Numéro de tour"
})

colonnes = [
    "Code du département",
    "Libellé du département",
    "Code de la commune",
    "Libellé de la commune",
    "Code B.Vote",
    "Inscrits",
    "Votants",
    "Exprimés",
    "Numéro de tour"
]

election_2014["Libellé du département"] = election_2014["Code du département"].map(departements)
election_2014 = election_2014[colonnes].drop_duplicates().copy()

election_2014["Année"] = 2014

election_2014["Abstentions"] = election_2014["Inscrits"] - election_2014["Votants"]
election_2014["Non-Exprimés"] = election_2014["Votants"] - election_2014["Exprimés"]

election_2014["% Abs/Ins"] = round(election_2014["Abstentions"] / election_2014["Inscrits"] * 100, 2)
election_2014["% Vot/Ins"] = round(election_2014["Votants"] / election_2014["Inscrits"] * 100, 2)

election_2014["% Non-Exp/Ins"] = round(election_2014["Non-Exprimés"] / election_2014["Inscrits"] * 100, 2)
election_2014["% Non-Exp/Vot"] = round(election_2014["Non-Exprimés"] / election_2014["Votants"] * 100, 2)

election_2014["% Exp/Ins"] = round(election_2014["Exprimés"] / election_2014["Inscrits"] * 100, 2)
election_2014["% Exp/Vot"] = round(election_2014["Exprimés"] / election_2014["Votants"] * 100, 2)

ordre = [
    "Code du département",
    "Libellé du département",
    "Code de la commune",
    "Libellé de la commune",
    "Inscrits",
    "Abstentions",
    "% Abs/Ins",
    "Votants",
    "% Vot/Ins",
    "Non-Exprimés",
    "% Non-Exp/Ins",
    "% Non-Exp/Vot",
    "Exprimés",
    "% Exp/Ins",
    "% Exp/Vot",
    "Année",
    "Numéro de tour"
]

election_2014 = election_2014[ordre]

election_2020["Non-Exprimés"] = election_2020["Blancs"].fillna(0) + election_2020["Nuls"].fillna(0)

election_2020["% Non-Exp/Ins"] = round(election_2020["Non-Exprimés"] / election_2020["Inscrits"] * 100, 2)
election_2020["% Non-Exp/Vot"] = round(election_2020["Non-Exprimés"] / election_2020["Votants"] * 100, 2)

election_2020 = election_2020[ordre]

election = pd.concat([election_2014, election_2020], ignore_index=True)

conditions = [
    election["Libellé de la commune"].isin(communes_grenoble),
    election["Libellé de la commune"].isin(communes_rennes),
    election["Libellé de la commune"].isin(communes_rouen),
    election["Libellé de la commune"].isin(communes_saint_etienne),
    election["Libellé de la commune"].isin(communes_montpellier)
]

choix = ["Grenoble", "Rennes", "Rouen", "Saint-Étienne", "Montpellier"]

election["Métropole"] = np.select(conditions, choix, default="Autre")

election.to_csv("elections_2014_2020.csv", index=False, encoding='utf-8-sig', sep=';')